# Adapting AlphaGenome to ATAC data with parameter-efficient finetuning

Demonstrates **LoRA (Low-Rank Adaptation)** for parameter-efficient fine-tuning of AlphaGenome.

**When to prefer LoRA over heads-only fine-tuning:**
Heads-only training adds a fresh random head on top of frozen backbone embeddings.
LoRA additionally inserts trainable low-rank matrices *inside* your head's linear projections,
giving them more expressive power at very low parameter cost.
Use LoRA when heads-only performance has plateaued and you need richer adaptation without
the memory and compute cost of full fine-tuning.

## 1. Install and import dependencies

In [ ]:
# Installation will take a few minutes
%pip install -q "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
%pip install -q git+https://github.com/google-deepmind/alphagenome.git
%pip install -q git+https://github.com/google-deepmind/alphagenome_research.git
%pip install -q alphagenome-ft
print("Done.")

In [ ]:
from __future__ import annotations

from pathlib import Path
import gzip
import shutil
import urllib.request

import jax
import jax.numpy as jnp
import haiku as hk
from alphagenome.models import dna_output

from alphagenome_ft import (
    create_model_with_heads,
    register_custom_head,
    CustomHead,
    CustomHeadConfig,
    CustomHeadType,
    lora,
    parameter_utils,
)
from alphagenome_ft.finetune.config import load_targets_config, prepare_head_specs
from alphagenome_ft.finetune.data import (
    prepare_intervals_from_fold,
    BigWigDataModule,
    build_fasta_index,
)
from alphagenome_ft.finetune.train import train

## 2. Define the user config

Edit the following config values to specify data paths and training hyperparameters.

In [ ]:
# Core input files
PROJECT_DIR = Path("/oak/stanford/groups/akundaje/abuen/personal/alphagenome_ft")
FASTA_PATH = Path("/oak/stanford/groups/akundaje/abuen/public/genome/hg38.genome.fa")

# Head identifier – used in checkpoints and as the BigWig target key
HEAD_NAME = "lora_rna_head"

# Target BigWig files (must be listed in the same order as NUM_TRACKS)
TARGET_BIGWIGS = [
    PROJECT_DIR / "data" / "k562_atac.bw"
]
NUM_TRACKS = len(TARGET_BIGWIGS)  # must match number of BigWig files above

# LoRA hyperparameters
# rank: dimension of the low-rank matrices A and B
# alpha: scaling factor; effective scale applied to the LoRA delta is alpha / rank
#        setting alpha == rank gives a scale of 1.0
LORA_RANK  = 8
LORA_ALPHA = 8.0

# Data options
FOLD        = "1"         # 0, 1, 2, 3
WINDOW_SIZE = 1_048_576   # 1 Mbp window

# Model options
ORGANISM      = "HOMO_SAPIENS"
MODEL_VERSION = "fold_1"  # fold_0, fold_1, fold_2, fold_3, all_folds

# Training options
BATCH_SIZE       = 1
NUM_EPOCHS       = 10
LEARNING_RATE    = 3e-4
WEIGHT_DECAY     = 1e-2
SEED             = 42
SHUFFLE          = True
DROP_LAST        = False
MAX_TRAIN_STEPS  = None
VERBOSE          = True

# Checkpointing / early stopping
CHECKPOINT_DIR          = PROJECT_DIR / "checkpoints/lora_demo"
BEST_METRIC             = "valid_loss"
BEST_METRIC_MODE        = "min"
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.0

## 2. Optional data download

Download genome FASTA file and the K562 pseudobulked ATAC files from `alphagenome_ft/data` on GitHub when missing.

In [ ]:
ENABLE_DATA_DOWNLOAD  = False
ENABLE_FASTA_DOWNLOAD = False

GITHUB_DATA_BASE = "https://raw.githubusercontent.com/genomicsxai/alphagenome_ft/main/data"
DEMO_FILES = ["k562_atac.bw"]
FASTA_DOWNLOAD_URL = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

if ENABLE_DATA_DOWNLOAD:
    for filename in DEMO_FILES:
        dst = PROJECT_DIR / filename
        if dst.exists():
            print(f"Skip existing: {dst}")
            continue
        src = f"{GITHUB_DATA_BASE}/{filename}"
        print(f"Downloading: {src} -> {dst}")
        urllib.request.urlretrieve(src, dst)
else:
    print("Core data download is disabled.")

if ENABLE_FASTA_DOWNLOAD:
    if FASTA_PATH.exists():
        print(f"Skip existing: {FASTA_PATH}")
    else:
        gz_path = FASTA_PATH.with_suffix(FASTA_PATH.suffix + ".gz")
        print(f"Downloading: {FASTA_DOWNLOAD_URL} -> {gz_path}")
        urllib.request.urlretrieve(FASTA_DOWNLOAD_URL, gz_path)
        print(f"Decompressing: {gz_path} -> {FASTA_PATH}")
        with gzip.open(gz_path, "rb") as src, FASTA_PATH.open("wb") as dst:
            shutil.copyfileobj(src, dst)
        gz_path.unlink()
else:
    print("FASTA download is disabled.")

FASTA_INDEX_PATH = Path(f"{FASTA_PATH}.fai")
if FASTA_PATH.exists() and not FASTA_INDEX_PATH.exists():
    print(f"Building FASTA index: {FASTA_INDEX_PATH}")
    build_fasta_index(FASTA_PATH)
elif FASTA_INDEX_PATH.exists():
    print(f"Found FASTA index: {FASTA_INDEX_PATH}")
else:
    print("Skip FASTA indexing because FASTA is missing.")

## 3. Validate inputs

In [ ]:
for path in [FASTA_PATH, FASTA_INDEX_PATH, *TARGET_BIGWIGS]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

if CHECKPOINT_DIR is not None:
    CHECKPOINT_DIR = Path(CHECKPOINT_DIR).resolve()
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Input validation passed.")

## 4. Define and register the LoRA head

`LoRALinear` wraps a single linear projection with two trainable low-rank matrices:

```
output = x @ W  +  (x @ A) @ B  *  (alpha / rank)
```

- **`W`** (`w`) — base weight, kept frozen after `freeze_except_lora`.
- **`A`** (`lora_a`) — maps input to the low-rank space; shape `(in_dim, rank)`.
- **`B`** (`lora_b`) — maps from low-rank back to output space; shape `(rank, out_dim)`.
  Initialised to zero so the adapter starts as a no-op and training begins from the
  frozen-backbone baseline.

The head below uses 1 bp embeddings from the decoder (`resolution=1`, shape `(B, S, 1536)`)
and applies a single `LoRALinear` to project down to `NUM_TRACKS` output channels.

In [ ]:
class LoRAHead(CustomHead):
    """Prediction head whose linear projection is equipped with LoRA adapters.

    Data flow:
        embeddings_1bp  (B, S, 1536)
            |  LoRALinear: x@W + (x@A)@B * (alpha/rank)
            v
        logits          (B, S, num_tracks)
            |  softplus
            v
        predictions     (B, S, num_tracks)

    Only ``lora_a`` and ``lora_b`` receive gradients during training;
    the base weight ``w`` is stopped via ``freeze_except_lora``.
    """
    def predict(self, embeddings, organism_index, **kwargs):
        x   = embeddings.get_sequence_embeddings(resolution=1)
        cfg = lora.LoRAConfig(rank=LORA_RANK, alpha=LORA_ALPHA)
        x   = lora.LoRALinear(self._num_tracks, cfg)(x)
        return {"predictions": jax.nn.softplus(x)}

    def loss(self, predictions, batch):
        preds   = predictions["predictions"]   # (B, S, num_tracks)
        targets = batch["targets"]             # (B, S, num_tracks)
        return {"loss": jnp.mean(jnp.square(preds - targets))}

register_custom_head(
    HEAD_NAME,
    LoRAHead,
    head_config=CustomHeadConfig(
        type=CustomHeadType.GENOME_TRACKS,
        output_type=dna_output.OutputType.ATAC,
        num_tracks=NUM_TRACKS,
    ),
)
print(f"Registered custom head: {HEAD_NAME!r}")

## 5. Build head specs and fold intervals

Specify the targets config. `source: custom` tells `prepare_head_specs` to look up
the head in the registry populated by `register_custom_head` above.

In [ ]:
TARGETS_CONFIG = {
    "heads": [
        {
            "id": HEAD_NAME,
            "source": "custom",
            "targets": [{"path": str(bw)} for bw in TARGET_BIGWIGS],
        }
    ]
}

config_dict = load_targets_config(TARGETS_CONFIG)
head_specs  = prepare_head_specs(config_dict)

intervals = prepare_intervals_from_fold(
    fold=FOLD,
    window_size=WINDOW_SIZE,
    organism=ORGANISM,
)

print("Heads:", [spec.head_id for spec in head_specs])
for split in ("train", "valid", "test"):
    print(f"{split}: {len(intervals.get(split, []))} intervals")

## 6. Initialize the model

In [ ]:
model = create_model_with_heads(
    MODEL_VERSION,
    heads=[HEAD_NAME],
)

print("Model ready.")

## 7. Inspect the backbone and LoRA parameter paths

Before freezing anything, inspect which parameters belong to the backbone vs. the LoRA
adapters.  `lora.get_lora_parameter_paths` finds every leaf named `lora_a` or `lora_b`;
`lora.count_lora_parameters` sums their element counts so you can verify the adapter
is sufficiently lightweight for fine-tuning.

In [ ]:
backbone_paths = parameter_utils.get_backbone_parameter_paths(model._params)
print(f"Backbone parameter arrays : {len(backbone_paths):,}")
print("Sample backbone paths:")
for p in backbone_paths[:6]:
    print("  ", p)
print()

In [ ]:
lora_paths = lora.get_lora_parameter_paths(model._params)
print(f"LoRA adapter arrays: {len(lora_paths)}")
for p in lora_paths:
    print("  ", p)
print()

total_params = parameter_utils.count_parameters(model._params)
lora_params  = lora.count_lora_parameters(model._params)
print(f"Total parameters : {total_params:>14,}")
print(f"LoRA parameters  : {lora_params:>14,}")
print(f"LoRA fraction    : {lora_params / total_params:.4%}")

## 8. Freeze the backbone and keep the LoRA adapters trainable

`freeze_except_lora` applies `jax.lax.stop_gradient` to every backbone parameter
**except** leaves named `lora_a` or `lora_b`.  This is equivalent to `freeze_backbone`
when LoRA adapters live in head paths (as they do here).

Frozen parameters still appear in the parameter tree; gradients flowing through them
are zeroed by `stop_gradient` so the optimizer never updates them.

In [ ]:
model._params = parameter_utils.freeze_except_lora(model._params)
print("Backbone frozen.  LoRA adapters (lora_a, lora_b) remain trainable.")

# Confirm the head (and its LoRA params) are still in the tree
head_paths = parameter_utils.get_head_parameter_paths(model._params)
print(f"\nHead parameter arrays: {len(head_paths)}")
for p in head_paths:
    print("  ", p)

## 9. Build the BigWig data module

In [ ]:
data_module = BigWigDataModule(
    intervals=intervals,
    fasta_path=FASTA_PATH,
    head_specs=head_specs,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE,
    drop_last=DROP_LAST,
)

print("Data module ready.")

## 10. (Optional) Smoke test settings

Use this to verify the full pipeline quickly before a long run.

In [ ]:
SMOKE_TEST = True
if SMOKE_TEST:
    NUM_EPOCHS      = 1
    MAX_TRAIN_STEPS = 20
    print("Smoke test enabled.")
else:
    print("Smoke test disabled.")

## 11. Train the model

`heads_only=True` labels backbone parameters as frozen inside the optimizer
(using `optax.set_to_zero()`) so no optimizer state is allocated for them.
The backbone freeze applied in step 8 is preserved; `train()` calls `freeze_backbone`
internally as a no-op on top of the existing stop-gradients.

In [ ]:
train(
    model,
    data_module,
    head_specs,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_epochs=NUM_EPOCHS,
    seed=SEED,
    max_train_steps=MAX_TRAIN_STEPS,
    heads_only=True,   
    checkpoint_dir=CHECKPOINT_DIR,
    organism=ORGANISM,
    best_metric=BEST_METRIC,
    best_metric_mode=BEST_METRIC_MODE,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    verbose=VERBOSE,
)